# Part 3: Add a Fabric IQ knowledge source

Microsoft Fabric is an end-to-end data platform with cloud services and tools for every data lifecycle stage—ingestion, preparation, storage, analysis, and visualization. Fabric IQ organizes OneLake data into an ontology (business concepts, relationships, and logic) so applications and agents can query data using consistent semantic context.

In this step, you'll create a multi-source knowledge base with a Fabric Ontology knowledge source.

> **Before you start:** Run `az login --tenant <your-tenant-id>` in a terminal before starting this notebook. This notebook uses `AzureCliCredential` to request delegated query-source tokens from your Azure CLI sign-in.

## Step 1: Set up environment variables

Load the configuration for your Azure AI Search, Azure OpenAI, and Fabric resources. The `.env` file in the project folder has the values needed for this part.

In [ ]:
import json
import os

import requests
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_ONTOLOGY_ID = os.environ["FABRIC_ONTOLOGY_ID"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

def azure_search_api_url(path: str) -> str:
    sep = "&" if "?" in path else "?"
    return f"{AZURE_SEARCH_SERVICE_ENDPOINT}{path}{sep}api-version={AZURE_SEARCH_API_VERSION}"

session = requests.Session()
session.headers.update({"api-key": AZURE_SEARCH_ADMIN_KEY, "Accept": "application/json"})


print("Environment variables loaded")

## Step 2: Get the Search query-source token

The Fabric Ontology source is queried at runtime using a delegated token. In this step, get a Search query-source token for the target tenant so the knowledge base can access Fabric IQ during retrieval.

This notebook uses `AzureCliCredential`, so the token comes from the Azure CLI account you signed in with before starting the notebook.

In [ ]:
from azure.identity import AzureCliCredential

query_source_credential = AzureCliCredential(tenant_id=AZURE_TENANT_ID, process_timeout=60)
QUERY_SOURCE_AUTH = query_source_credential.get_token("https://search.azure.com/.default").token
print("Acquired token for Azure AI Search query source")

## Step 3: Create three knowledge sources

For this knowledge base, you'll create three knowledge sources: the same two search-index sources used earlier, plus a Fabric Ontology knowledge source.

* `hrdocs-knowledge-source`: Points to the `hrdocs` search index
* `healthdocs-knowledge-source`: Points to the `healthdocs` search index
* `fabric-ontology-knowledge-source`: Points to the Fabric workspace and ontology used for structured operational data

In [ ]:
HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
FABRIC_KNOWLEDGE_SOURCE_NAME = "fabric-ontology-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    FABRIC_KNOWLEDGE_SOURCE_NAME,
]

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Health benefits documents"),
]:
    body = {
        "name": knowledge_source_name,
        "description": description,
        "kind": "searchIndex",
        "searchIndexParameters": {
            "searchIndexName": index_name,
            "sourceDataFields": [{"name": "uid"}, {"name": "snippet_parent_id"}, {"name": "blob_path"}, {"name": "snippet"}],
            "searchFields": [{"name": "snippet"}],
            "semanticConfigurationName": "semantic-configuration",
        },
    }
    r = session.put(
        azure_search_api_url(f"/knowledgesources('{knowledge_source_name}')"),
        json=body,
        headers={"Prefer": "return=representation"},
    )
    r.raise_for_status()

fabric_knowledge_source_body = {
    "name": FABRIC_KNOWLEDGE_SOURCE_NAME,
    "kind": "fabricOntology",
    "description": "Fabric Ontology knowledge source",
    "fabricOntologyParameters": {
        "fabricEndpoint": "https://api.fabric.microsoft.com",
        "workspaceId": FABRIC_WORKSPACE_ID,
        "ontologyId": FABRIC_ONTOLOGY_ID,
    },
}
r = session.put(
    azure_search_api_url(f"/knowledgesources('{FABRIC_KNOWLEDGE_SOURCE_NAME}')"),
    json=fabric_knowledge_source_body,
    headers={"Prefer": "return=representation"},
)
r.raise_for_status()
print("Knowledge sources created")

## Step 4: Create the multi-source + Fabric knowledge base

A knowledge base combines the knowledge sources with an LLM and instructions for how retrieval should behave.

For this notebook, the knowledge base uses an `outputMode` of `answerSynthesis` so the attached model can answer the question directly. It also uses a `retrievalReasoningEffort` of `low` so the model can help with query planning and source selection.

In [ ]:
KNOWLEDGE_BASE_NAME = "multisource-fabric-ontology-knowledge-base"

body = {
    "name": KNOWLEDGE_BASE_NAME,
    "description": "Multi-source knowledge base combining restored indexes and Fabric Ontology",
    "outputMode": "answerSynthesis",
    "retrievalReasoningEffort": {"kind": "low"},
    "knowledgeSources": [{"name": name} for name in KNOWLEDGE_SOURCE_NAMES],
    "models": [
        {
            "kind": "azureOpenAI",
            "azureOpenAIParameters": {
                "resourceUri": AZURE_OPENAI_ENDPOINT.rstrip("/") + "/",
                "apiKey": AZURE_OPENAI_KEY,
                "deploymentId": AZURE_OPENAI_CHATGPT_DEPLOYMENT,
                "modelName": AZURE_OPENAI_CHATGPT_MODEL_NAME,
            },
        }
    ],
    "retrievalInstructions": "Use Fabric Ontology for structured operational data. Use the restored indexes for HR and health policy documents.",
}
r = session.put(
    azure_search_api_url(f"/knowledgebases('{KNOWLEDGE_BASE_NAME}')"),
    json=body,
    headers={"Prefer": "return=representation"},
)
r.raise_for_status()
print("Knowledge base created")

## Step 5: Query the knowledge base

Ask a question that combines structured Fabric Ontology data with reasoning from the attached chat model.

The knowledge base uses agentic retrieval to decide when to query the search indexes and when to query the Fabric Ontology source.

In [ ]:
from IPython.display import Markdown, display

question = "Using the Zava DIY Fabric ontology Product entity, list a few product names with category and stockLevel. Then explain how stockLevel can help with inventory planning."

retrieve_body = {
    "includeActivity": True,
    "messages": [
        {
            "role": "user",
            "content": [{"type": "text", "text": question}],
        }
    ],
    "knowledgeSourceParams": [
        {
            "kind": "searchIndex",
            "knowledgeSourceName": HR_KNOWLEDGE_SOURCE_NAME,
            "includeReferences": True,
            "includeReferenceSourceData": True,
            "alwaysQuerySource": False,
        },
        {
            "kind": "searchIndex",
            "knowledgeSourceName": HEALTH_KNOWLEDGE_SOURCE_NAME,
            "includeReferences": True,
            "includeReferenceSourceData": True,
            "alwaysQuerySource": False,
        },
        {
            "kind": "fabricOntology",
            "knowledgeSourceName": FABRIC_KNOWLEDGE_SOURCE_NAME,
            "includeReferences": True,
            "includeReferenceSourceData": True,
            "alwaysQuerySource": True,
        },
    ],
    "maxRuntimeInSeconds": 180,
}
r = session.post(
    azure_search_api_url(f"/knowledgebases('{KNOWLEDGE_BASE_NAME}')/retrieve"),
    json=retrieve_body,
    headers={"x-ms-query-source-authorization": QUERY_SOURCE_AUTH},
    timeout=240,
)
print(f"{r.status_code} {r.reason}")
if not r.ok:
    print(r.text)
else:
    result = r.json()
    display(Markdown(result["response"][0]["content"][0]["text"]))

### Review the references

The next cell prints the raw `references` array returned by the knowledge base so you can inspect which sources contributed to the answer.

In [ ]:
print(json.dumps(result.get("references", []), indent=2))

## Bonus: Query from the Copilot CLI

Every knowledge base exposes an MCP server endpoint, which can be added to any MCP-compatible client like VS Code or GitHub Copilot CLI.
If you want to try out querying the knowledge base from the Copilot CLI, use the generated MCP configuration below and follow the steps in the [Copilot CLI sidequest](copilot-cli-sidequest.md).

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
headers = {"api-key": AZURE_SEARCH_ADMIN_KEY}
config = {"mcpServers": {KNOWLEDGE_BASE_NAME: {"type": "http", "url": mcp_url, "headers": headers}}}
print(json.dumps(config, indent=2))

➡️ Continue to: [Part 4: Add Work IQ](part4-work-iq-to-kb.ipynb).